# Setup

In [1]:
import json
from pathlib import Path
from langchain.text_splitter import RecursiveCharacterTextSplitter
import re

CHUNK_SIZE = 4000

# Chunker

In [ ]:
# carrega páginas HTML parseadas
with open("../data/parsed/pages_parsed.json", "r", encoding="utf-8") as f:
    pages_parsed = json.load(f)

# carrega PDFs parseados
with open("../data/parsed/pdfs_parsed.json", "r", encoding="utf-8") as f:
    pdfs_parsed = json.load(f)

print(f"Páginas HTML: {len(pages_parsed)}")
print(f"PDFs: {len(pdfs_parsed)}")

Páginas HTML: 2029
PDFs: 595


In [3]:
# configura o splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_SIZE*0.2,
    length_function=len,
)

def chunk_document(text, metadata):
    # documento curto: 1 chunk direto
    if len(text) <= CHUNK_SIZE:
        return [{"text": text, **metadata}]
    
    # documento longo: divide com overlap
    parts = splitter.split_text(text)
    return [{"text": part, **metadata} for part in parts]

# transforma URLs do Google Drive para formato de visualização
def to_drive_view_url(url):
    if "drive.google.com/uc" in url:
        match = re.search(r"id=([a-zA-Z0-9_-]+)", url)
        if match:
            return f"https://drive.google.com/file/d/{match.group(1)}/view"
    return url

In [4]:
chunks = []

# processa páginas HTML
for page in pages_parsed:
    metadata = {
        "source_url": page["source_url"],
        "title": page["title"],
        "type": "html",
        "published_at": page.get("published_at")
    }
    chunks.extend(chunk_document(page["text"], metadata))

# processa PDFs (ignora escaneados)
for pdf in pdfs_parsed:
    if pdf["is_scanned"]:
        continue
    metadata = {
        "source_url": to_drive_view_url(pdf["source_url"]),
        "title": pdf["title"],
        "type": "pdf",
        "published_at": pdf.get("published_at")
    }
    chunks.extend(chunk_document(pdf["text"], metadata))

print(f"Total de chunks: {len(chunks)}")

# salva chunks em disco
Path("../data/chunks").mkdir(parents=True, exist_ok=True)

with open("../data/chunks/chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"Salvo: {len(chunks)} chunks em data/chunks/chunks.json")

Total de chunks: 5783
Salvo: 5783 chunks em data/chunks/chunks.json
